# Tutorial 04: Dynamics and control of a Simple Pendulum

In this tutorial, you will implement forward and inverse dynamics of a simple pendulum system and study its applications in simulation as well as control.

<div style="display: flex; justify-content: space-around;">
    <div><img src="media/crane.jpg" width="400"></div>
    <div><img src="media/swing.gif" width="400"></div>
</div>

**Pre-requisites**

Knowledge of kinematic and dynamic models of simple robots, basic knowledge of integration schemes, underactuated robots.

**Goals**

Exploring the modelling and some control strategies for underactuated robots.

This notebook is organized as follows:

    1. Recap: Lagrangian Derivation of Equations of Motion (EOM)
    2. Derive the Solution to Forward and Inverse Dynamics
    3. Simulation using Forward Dynamics
    4. Gravity Compensation Control using Inverse Dynamics
    5. Energy Shaping Control to perform a swing-up
    6. Experiments on Pendulum Hardware (teacher only)
      a. Free-fall experiment
      b. Gravity compensation 
      c. Energy shaping



## 1. Recap: Lagrangian Derivation of EOM

Consider a simple pendulum (undamped) as shown in the figure below. In the following, we will derive the EOM of the system using Python Symbolic Package called sympy.

<img src="media/pendulum_undamped_axes.png" width="400">

Physical parameters of the pendulum:
 * Mass: m
 * Length: l
 * Gravity: g

In [ ]:
import sympy 
import numpy as np

# Creates symbolic variable for physical parameters of the pendulum
m = sympy.Symbol('m')  # mass
l = sympy.Symbol('l')  # length
g = sympy.Symbol('g')  # gravity

The generalized coordinates and their derivatives are:
 * Angle: $q = \theta$
 * Angular Velocity: $\dot q = \dot{\theta}$
 * Angular Acceleration: $\ddot q = \ddot{\theta}$

In [ ]:
t = sympy.Symbol('t')  # Creates symbolic variable t for time

# Creates symbolic variables theta, thetadot and thetaddot
th = sympy.Function('theta')(t) 
th_dot = sympy.Function('theta_dot')(t) 
th_ddot = sympy.Function('theta_ddot')(t) 

 Recall that the Lagrangian of a mechanical system is defined as:
 $$
 L = K - V
 $$

In [ ]:
# Position Equation: r = [x, y]
r = sympy.Matrix([l * sympy.sin(th), -l * sympy.cos(th)])
print('\nPosition')
display(r)

# Velocity Equation: d/dt(r) = [dx/dt, dy/dt]
v = sympy.Matrix([r[0].diff(t), r[1].diff(t)])
print('\nVelocity')
display(v)

# Develop the Lagrangian of the system
# Kinetic Energy
K = (1/2 * v.T * m @ v )[0]
print('\nKinetic energy')
display(sympy.simplify(K))

# Potential Energy 
V = m * g * r[1]
print('\nPotential energy')
display(sympy.simplify(V))

# Lagrangian
L = K - V
print('\nLagrangian')
display(sympy.simplify(L))

 The Euler-Lagrange equations can be used to derive the Equations of Motion (EOM) of the system:
 $$
 \frac{d}{dt}\frac{\partial L}{\partial \dot{\mathbf{q}}} - \frac{\partial L}{\partial \mathbf{q}} = \tau
 $$
 The EOM of any mechanical system can be written in the canonical form:
 $$
 \mathbf{M} (\mathbf{q}) \ddot{\mathbf{q}} +  \mathbf{C} (\mathbf{q}, \dot{\mathbf{q}}) = \tau
 $$

In [ ]:
# Lagrange Terms:
dL_dth = L.diff(th)
dL_dth_dt = L.diff(th.diff(t)).diff(t)

# Euler-Lagrange Equations: dL/dq - d/dt(dL/ddq) = 0
eom = sympy.simplify(dL_dth_dt - dL_dth)

# Replace Time Derivatives and Functions with Symbolic Variables:
replacements = [(th.diff(t).diff(t), th_ddot), (th.diff(t),  th_dot)]
# Note: Replace in order of decreasing derivative order
eom = eom.subs(replacements)  # Use Subs method to substitute in the Symbolic Variables
print('tau')
display(eom)

Additionally, we may add a damping term to account for the dissipation of energy in the joint of tyhe pendulum. We can call this term $\tau_{d}$.

In this case, we model the damping as torque proportional and opposite in direction to $\dot{\mathbf{q}}$.

$$
\tau_{d} = -\mathbf{b} \dot{\mathbf{q}}
$$

$$
\mathbf{M} (\mathbf{q}) \ddot{\mathbf{q}} +  \mathbf{C} (\mathbf{q}, \dot{\mathbf{q}}) = \tau + \tau_{d}
$$

In [ ]:
b = sympy.Symbol('b')  # damping

eom = eom + b*th.diff(t)
eom = eom.subs(replacements)
print("tau = ", eom)

## 2. Forward and Inverse Dynamics

**Inverse Dynamics** is defined as the problem of solving for the generalized forces given the position, velocity and acceleration of the mechanical system in generalized coordinates. 
$$
\tau = \text{idyn}(\mathbf{q}, \dot{\mathbf{q}}, \ddot{\mathbf{q}}) = \mathbf{M} (\mathbf{q}) \ddot{\mathbf{q}} +  \mathbf{C} (\mathbf{q}, \dot{\mathbf{q}}) - \tau_{d}
$$

In [ ]:
def inverse_dynamics(self, th, th_dot, th_ddot):
    """
    return torque acting on the revolute joint (tau) in terms of inputs
    use self.m, self.g, self.l, and self.b if needed
    """
    m = self.m
    g = self.g
    l = self.l
    b = self.b
    
    tau = th_ddot*m*l*l + m*g*l*np.sin(th) + b*th_dot # Type Here!

    return tau

**Forward Dynamics** is defined as the problem of solving for the generalized acceleration of the mechanical system, given generalized position, velocity and forces acting on the system. 

$$
\ddot{\mathbf{q}} = \text{fdyn}(\mathbf{q}, \dot{\mathbf{q}}, \tau) = \mathbf{M}^{-1} (\mathbf{q}) \left( \tau + \tau_d -  \mathbf{C} (\mathbf{q}, \dot{\mathbf{q}}) \right)
$$

Note: Forward Dynamics of a mechanical system is a **second-order ordinary differential equation (ODE)**. 

In [ ]:
def forward_dynamics(self, th, th_dot, tau):
    """
    Return acceleration from current position, velocity and torque.
    """
    m = self.m
    g = self.g
    l = self.l
    b = self.b

    th_ddot = (tau - m*g*l*np.sin(th) - b*th_dot) / (m*l*l) # Type Here!
    
    return th_ddot

## 3. Simulation using Forward Dynamics

The state space of the system is the tuple of $\theta$ and its time derivative $\dot{\theta}$:
$$
\mathbf{x} = (\mathbf{q}, \dot{\mathbf{q}})
$$
The state space form of a dynamical system is given by:
$$
\dot{\mathbf{x}} = \mathbf{f}(\mathbf{x}, \mathbf{u})
$$
where $\mathbf{u}$ is the vector of control input. Note that the above form is a **first-order ODE**. 

Using the forward dynamics, one can deduce the first-order ODE of any mechanical system using the following technique:
$$
\dot{\mathbf{x}} = \mathbf{f}(\mathbf{x}, \mathbf{u}) = 
\begin{bmatrix}
\dot{\mathbf{q}}\\ 
\mathbf{M}^{-1} (\mathbf{q}) \left( \tau + \tau_d -  \mathbf{C} (\mathbf{q}, \dot{\mathbf{q}}) \right)
\end{bmatrix}
$$

The solution of this first-order ODE can be used to develop a dynamics simulator for a rigid multi-body system. While this can not be solved analytically, even for simpler systems, we can use numerical schemes to solve the ODEs.

Given an initial state $\mathbf{x}_0$, input  $\mathbf{u}$ and time step $dt$, we can predict the behavior of the mechanical system in the next time step using a numerical integration scheme. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math 

%matplotlib inline
from IPython.display import HTML

from pendulum_plant_t04 import PendulumPlant, plot_timeseries

The cell below constructs a pendulum object with the specified dynamic parameters. For now, our damping will be zero.

In [ ]:
mass = 0.06      # mass at the end of the pendulum [kg]
length = 0.1    # length of the rod [m]
damping = 0.0   # viscious friction [kg m/s]
gravity = 9.81  # gravity [m/s²]
inertia = mass*length*length   # inertia of the pendulum [kg m²]
torque_limit = np.inf  # torque limit of the motor, here we assume the motor can produce any torque

pendulum = PendulumPlant(mass=mass,
                         length=length,
                         damping=damping,
                         gravity=gravity,
                         inertia=inertia,
                         torque_limit=torque_limit)

# Add the forward dynamics method you implemented to the pendulum plant class 
PendulumPlant.forward_dynamics = forward_dynamics

# Add the inverse dynamics method you implemented to the pendulum plant class 
PendulumPlant.inverse_dynamics = inverse_dynamics

The first step we must take to design our integration scheme is to get our ODE.

In [ ]:
def f(self, t, x, tau):

    """
    Computes the integrand of the equations of motion.
    Hint: use the self.forward_dynamics function to compute acceleration
    """
    accn = self.forward_dynamics(x[0], x[1], tau) # Type Here!
    xdot = np.array([x[1], accn])
    return xdot # Return xdot

PendulumPlant.f = f

### 3.1 Explicit Euler integration scheme

Using Euler integration scheme leads to the following update rule: 
$$
\mathbf{x}[n+1] = \mathbf{x}[n] + \mathbf{f}(\mathbf{x}[n], \mathbf{u}[n]) dt
$$

Complete the following function implementing an explicit Euler integration scheme:

In [ ]:
def euler_integrator(self, t, x, dt, tau):

    ## Implement an Explicit Euler integration scheme

    integ = self.f(t, x, tau)
    x_new = x + dt*integ
    return x_new

PendulumPlant.euler_integrator = euler_integrator

In order to simulate and animate a motion of the pendulum you can call the simulate_and_animate class method. The arguments are:

    - t0: the initial time
    - x0: the initial state vector containing position and velocity of the pendulum [th, th_dot]
    - tf: the final time
    - dt: the integration timestep
    - controller: controller object providing torques for the motor (introduced later)
    - integrator: the integration method. options are "euler" and "runge_kutta"
    
The method returns the trajectory split in to time (T), state (X) and torques (U) as well as an animation object.

Note: If you do not need the animation you can also call the method "simulate" with the same parameters. The method returns T, X, U but no animation object.

In [ ]:
%%capture
Tsim_euler, Xsim_euler, Usim_euler, anim_euler = pendulum.simulate_and_animate(
              t0=0.0,
              x0=[0.78, 0.0],
              tf=5.0,
              dt=0.002,
              controller=None,
              integrator="euler",
              anim_dt = 0.04)

to show the animation in this notebook call (may take around a minute)

In [ ]:
HTML(anim_euler.to_html5_video())

You can use the plot_timeseries function to plot the trajectories (position, velocity and torque). Of course you can also plot these in any other way you like.

In [ ]:
plot_timeseries(Tsim_euler, Xsim_euler, Usim_euler)

As you can observe, the video rendering in Jupyter is quite slow. But simulating the pendulum is not. Hence, use the simulate() method instead which suppresses the animation but still solves the ODE for you.  

In [ ]:
Tsim_euler_s, Xsim_euler_s, Usim_euler_s = pendulum.simulate(
              t0=0.0,
              x0=[0.78, 0.0],
              tf=10.0,
              dt=0.002,
              controller=None,
              integrator="euler")
plot_timeseries(Tsim_euler_s, Xsim_euler_s, Usim_euler_s)

### 3.2 Fourth Order Runge-Kutta integration scheme

Using the fourth order Runge-Kutta integration scheme leads to the following update rule: 
$$
\mathbf{x}[n+1] = \mathbf{x}[n] + \frac{dt}{6}(\mathbf{k}_1 + 2\mathbf{k}_2 + 2\mathbf{k}_3 + \mathbf{k}_4)
$$

$$
t_{n+1} = t_n + dt
$$

where

$$
\mathbf{k}_1  =\mathbf{f}(t_{n}, \mathbf{x}_{n})
$$

$$
\mathbf{k}_2  =\mathbf{f}(t_{n} + \frac{dt}{2}, \mathbf{x}_{n} + dt \frac{\mathbf{k}_1}{2})
$$

$$
\mathbf{k}_3  =\mathbf{f}(t_{n} + \frac{dt}{2}, \mathbf{x}_{n} + dt \frac{\mathbf{k}_2}{2})
$$

$$
\mathbf{k}_4  =\mathbf{f}(t_{n} + dt, \mathbf{x}_{n} + dt \mathbf{k}_3)
$$

Complete the following function implementing an fourth order Runge-Kutta integration scheme:

In [ ]:
def runge_integrator(self, t, x, dt, tau):
    
    k1 = self.f(t, x, tau)
    k2 = self.f(t + 0.5*dt, x + 0.5*dt*k1, tau)
    k3 = self.f(t + 0.5*dt, x + 0.5*dt*k2, tau)
    k4 = self.f(t + dt, x + dt*k3, tau)
    integ = (k1 + 2*(k2 + k3) + k4) / 6.0

    x_new = x + dt*integ

    return x_new
    
PendulumPlant.runge_integrator = runge_integrator

In [ ]:
%%capture
Tsim_RK4, Xsim_RK4, Usim_RK4, anim_RK4 = pendulum.simulate_and_animate(
              t0=0.0,
              x0=[0.78, 0.0],
              tf=5.0,
              dt=0.002,
              controller=None,
              integrator="runge_kutta",
              anim_dt = 0.04)

In [ ]:
HTML(anim_RK4.to_html5_video())

In [ ]:
Tsim_RK4, Xsim_RK4, Usim_RK4 = pendulum.simulate(
              t0=0.0,
              x0=[np.pi, 0.0],
              tf=10.0,
              dt=0.001,
              controller=None,
              integrator="euler")
plot_timeseries(Tsim_RK4, Xsim_RK4, Usim_RK4)

## Think-Pair-Share (15 min)

Try to play around with different initial conditions (x0), time step (dt), total simulation time (tf) choices! What do you observe?

1. Which integrator do you prefer?
2. What are the trade-offs of smaller vs bigger time steps?
3. What are the most interesting initial conditions? Can you already identify fixed points of the systems? What are attractors and repellors? Try adding damping to the simulation by modifying b in the definition of pendulum.
4. How accurate do you find these simulations? 

## 4. Gravity Compensation Control using Inverse Dynamics

### 4.1 Gravity Compensation

The gravity compensating controller cancels out the gravitational effect acting on the mass of the pendulum. With gravity compensation active, the pendulum will behave as it would in space.

The cell below shows the very basic structure of a controller class that the simulation method of the pendulum plant expects. Especially interesting is the get_control_output method. This method is called at every simulation step. Here the controller receives the current state of the pendulum and can compute a torque from that state, resulting in a reactive controller.

Please fill in the get_control method below, so that the returned torque cancels out the gravity effects acting on the pendulum mass.

In [ ]:
class GravityCompController():
    def __init__(self, mass, length, gravity):
        self.g = gravity        # gravitational acceleration on earth
        self.m = mass           # mass at the end of the pendulum
        self.l = length         # length of the rod

    def get_control_output(self, x):
        # compensate gravity with input torque
        m = self.m
        g = self.g
        l = self.l
        # Angle theta is available to you as a part of the state vector
        theta = x[0]
        
        ## Your Code Here        
        tau = m * g * l * np.sin(theta)  # Type Here! In terms of local variables m, g and l
        ##
        return tau

Next, we can initialize a controller object

In [ ]:
grav_con = GravityCompController(mass=mass,
                                 length=length,
                                 gravity=gravity)

and start a simulation with this controller

In [ ]:
Tsim_RK4, Xsim_RK4, Usim_RK4 = pendulum.simulate(
              t0=0.0,
              x0=[np.pi/2, 0.0],
              tf=10.0,
              dt=0.01,
              controller=grav_con,
              integrator="runge_kutta")
plot_timeseries(Tsim_RK4, Xsim_RK4, Usim_RK4)

If you implemented the torque response correctly, the pendulum should hold still at half height.

To see that the controller is actually doing something you can initialize the controller e.g. with 0.95*gravity and see the pendulum slowly falling down.

### 4.2 PID + Gravity compensation

Let's say we want to use gravity compensation to achieve a smooth swing-up motion with our pendulum. 

In [ ]:
class GravityPIDController():
    def __init__(self, mass, length, gravity, torque_limit, Kp, Ki, Kd, dt, grav_con):
        self.g = gravity        # gravitational acceleration on earth
        self.m = mass           # mass at the end of the pendulum
        self.l = length         # length of the rod
        
        self.Kp = Kp
        self.Ki = Ki
        self.Kd = Kd
        self.torque_limit = torque_limit
        
        self.dt = dt

        self.GC = grav_con # Gravity compensation controller
        
        self.errors = []

    def set_goal(self, goal=[np.pi, 0.0]):
        self.goal = goal

    def get_control_output(self, x):
        # Extract current state variables
        theta = x[0]  # current angle of the pendulum

        # Compute the error
        error = self.goal[0] - theta

        # Store the current error for the next iteration
        self.errors.append(error)

        # Compute the integral of the error
        integral = np.sum(self.errors) * self.dt

        # Compute the derivative of the error
        if len(self.errors) > 1:
            derivative = (self.errors[-1] - self.errors[-2]) / self.dt
        else:
            derivative = 0.0

        # Compute the control output
        tau = self.Kp * error + self.Ki * integral + self.Kd * derivative # PID version
        #tau = self.Kp * error - self.Kd * x[1] # PD version

        
        
        # add gravity compensation
        #tau += self.m * self.g * self.l * np.sin(theta)
        tau += self.GC.get_control_output(x)
        tau = np.clip(tau,-self.torque_limit, self.torque_limit)
        
        return tau

In [ ]:
# Gravity PID controller instance
grav_pid_con = GravityPIDController(mass, length, gravity, torque_limit = 0.14,
                                    Kp=0.01, Ki=0.001, Kd=0.001, dt=0.01, grav_con = grav_con)
grav_pid_con.set_goal([np.pi, 0.0])

In [ ]:
%%capture
Tsim_pid_gc, Xsim_pid_gc, Usim_pid_gc, anim_pid_gc = pendulum.simulate_and_animate(
              t0=0.0,
              x0=[0.001, 0.0],
              tf=10.0,
              dt=0.01,
              controller=grav_pid_con,
              integrator="runge_kutta",
              anim_dt = 0.04)

In [ ]:
HTML(anim_pid_gc.to_html5_video())

In [ ]:
plot_timeseries(Tsim_pid_gc, Xsim_pid_gc, Usim_pid_gc)

In [ ]:
Tsim_pid_gc_s, Xsim_pid_gc_s, Usim_pid_gc_s = pendulum.simulate(
              t0=0.0,
              x0=[0.001, 0.0],
              tf=10.0,
              dt=0.01,
              controller=grav_pid_con,
              integrator="runge_kutta")
plot_timeseries(Tsim_pid_gc_s, Xsim_pid_gc_s, Usim_pid_gc_s)

## Think-Pair-Share (5 min)

What happens if you modify the instance of the gravity PID controller? 

- Try adding a torque limit of 0.03 Nm, for instance. What do you see? Do you reach the desired position?
- Play around with the gains of the PID. What is the lowest torque limit for which the pendulum reaches the desired position?

## 5. Energy Shaping

For many underactuated systems, one very important quantity is the total energy in the system. In these systems, the amount of energy we can provide to our system at any given instant is limited. This may make it hard to reach one state that is energetically far from the starting point. One way we can accumulate the necessary energy is supply it progressively and strategically to the system. 

Total Energy (TE) = Kinetic Energy (KE) + Potential Energy (PE)

- Compute the total energy (TE) of the pendulum for the goal state of ($\pi$, 0).
- Use the TE to create an energy shaping control to go from state (0.001, 0) to ($\pi$, 0). *Hint:* The controller should apply torque in the direction that increases the TE. If the $TE_{current}$ < $TE_{desired}$ or it should apply torque in the direction that decreases the TE if the $TE_{current}$ > $TE_{desired}$.

To make it slightly more challenging, we add slight damping to the pendulum.

In [ ]:
pendulum.b = 0.0004 # Damping in the pendulum plant instance

In [ ]:
class EnergyShapingController():
    def __init__(self, mass, length, damping, torque_limit, gravity, K):
        self.K = K
        self.m = mass
        self.l = length
        self.g = gravity
        self.b = damping
        self.torque_limit = torque_limit

    def get_control_output(self, x):
        # Create a Energy Shaping Controller for swing up of simple pendulum 
        Ted = self.m * self.l * self.g  # desired energy
        Tec = (self.m * (self.l * x[1])**2)/2 - self.m * self.l * self.g * np.cos(x[0]) # current energy
        tau_es = -self.K * x[1] * (Tec - Ted) + self.b*x[1]
        ##        
        tau = np.clip(tau_es,-self.torque_limit, self.torque_limit) # Enforce limit on allowed torque
        
        return tau

In [ ]:
energy_shaping_controller = EnergyShapingController(mass=mass, length=length, damping = 0.0004, torque_limit=0.02, gravity=gravity, K=0.1)

In [ ]:
%%capture
T, X, U, anim = pendulum.simulate_and_animate(
              t0=0.0,
              x0=[0.001, 0.0],
              tf=10.0,
              dt=0.01,
              controller=energy_shaping_controller,
              integrator="runge_kutta",
              anim_dt = 0.04)

In [ ]:
HTML(anim.to_html5_video())

In [ ]:
plot_timeseries(T, X, U)

In [ ]:
Tsim_es, Xsim_es, Usim_es = pendulum.simulate(
              t0=0.0,
              x0=[0.001, 0.0],
              tf=10.0,
              dt=0.01,
              controller=energy_shaping_controller,
              integrator="runge_kutta")
plot_timeseries(Tsim_es, Xsim_es, Usim_es)


# Export the data to csv file
simulation_csv_data_es = np.array([Tsim_es, np.asarray(Xsim_es).T[0], np.asarray(Xsim_es).T[1],Usim_es]).T
np.savetxt("es_simulation_data_es.csv", simulation_csv_data_es, delimiter=',', header="time,pos,vel,tau", comments="")

In [ ]:
# Reshape the values
if len(pendulum.x_values)>2:
    pendulum.x_values = np.array(pendulum.x_values).T.tolist()
pendulum.plot_energy()

## Think-Pair-Share (10 min)

- Is this controller able to stabilize the top-most positions once it reaches there? Why/Why not?
- What is the lowest torque limit with which you can reach the desired position in within 10 seconds from the start of the simulation?

Robustness is a quality which is appreciated in controllers. Robust controllers will perform well even in cases in which there is some degree of uncertainty.
- Robustness: Try simulating different starting configurations, masses, and damping values. Do not change the controller parameters. How does the controller respond? How robust would you say this controller is?

## 6. Experiments on Pendulum Hardware

The following image shows a hardware prototype of an active pendulum system actuated with a small direct drive actuator (GL35 from T-motors). The control electronics are provided by MAB robotics. Through cloudpendulum, we can apply our designed controllers on the real system.

<img src="media/pendulum_hardware.jpeg" width="400">

To activate the pendulum, run the cell below! This command establishes the connection to the pendulum hardware via USB2CAN interface. 

**Note:** Once the kernel has been started, this cell should only be executed once. If you execute the cell twice, you will get an error. If that happens, please shutdown the kernel and restart it once again. 

In [ ]:
user_token = ""

## 6a. Free-fall Experiment

In [ ]:
x0 = np.pi/2
tf = 5 # Final time (s)
dt = 0.01 # Time step (s)
Treal_ff, Xreal_ff, Ureal_ff, Ureal_des, exp_video_url, exp_video_path = pendulum.run_on_hardware_cloud(user_token = user_token, 
                                                                                        tf = tf,
                                                                                        dt = dt,
                                                                                        x0 = x0,
                                                                                        controller=None
                                                                                        )
print('Finished!')
# Measured Position
plt.figure
plt.plot(Treal_ff, np.asarray(Xreal_ff).T[0])
plt.xlabel("Time (s)")
plt.ylabel("Position (rad)")
plt.title("Position (rad) vs Time (s)")
plt.show()

# Measured Velocity
plt.figure
plt.plot(Treal_ff, np.asarray(Xreal_ff).T[1])
plt.xlabel("Time (s)")
plt.ylabel("Velocity (rad/s)")
plt.title("Velocity (rad/s) vs Time (s)")
plt.show()

# Measured Torque
plt.figure
plt.plot(Treal_ff, Ureal_ff)
plt.xlabel("Time (s)")
plt.ylabel("Torque (Nm)")
plt.title("Torque (Nm) vs Time (s)")
plt.show()

# Export the data to csv file
measured_csv_data = np.array([Treal_ff, np.asarray(Xreal_ff).T[0], np.asarray(Xreal_ff).T[1],Ureal_ff]).T
np.savetxt("ff_measured_data.csv", measured_csv_data, delimiter=',', header="time,pos,vel,tau", comments="")

# Show video
from IPython.display import Video
Video(exp_video_path)

**Question: How does the behavior of the pendulum hardware differ from your simulation? What is missing in our previous plant model?**

## 6b. Gravity Compensation Experiment

Now we will test the gravity compensation controller that we developed directly on the real hardware. If your controller implementation is correct, you should be able to bring the pendulum by hand to any position in a quasi-static manner and it should hold itself. 

In [ ]:
x0 = np.pi/2
tf = 5 # Final time (s)
dt = 0.01 # Time step (s)
Treal_gc, Xreal_gc, Ureal_gc, Ureal_gc_des, exp_video_url, exp_video_path = pendulum.run_on_hardware_cloud(user_token = user_token, 
                                                                                        tf = tf,
                                                                                        dt = dt,
                                                                                        x0 = x0,
                                                                                        controller = grav_con
                                                                                                       )
print('finished!')                                                                      
# Measured Position
plt.figure
plt.plot(Treal_gc, np.asarray(Xreal_gc).T[0])
plt.xlabel("Time (s)")
plt.ylabel("Position (rad)")
plt.title("Position (rad) vs Time (s)")
plt.show()

# Measured Velocity
plt.figure
plt.plot(Treal_gc, np.asarray(Xreal_gc).T[1])
plt.xlabel("Time (s)")
plt.ylabel("Velocity (rad/s)")
plt.title("Velocity (rad/s) vs Time (s)")
plt.show()

# Measured Torque
plt.figure
plt.plot(Treal_gc, Ureal_gc)
plt.plot(Treal_gc, Ureal_gc_des)
plt.xlabel("Time (s)")
plt.ylabel("Torque (Nm)")
plt.legend(['Measured Torque', 'Desired Torque'])
plt.show()

# Export the data to csv file
measured_csv_data = np.array([Treal_ff, np.asarray(Xreal_ff).T[0], np.asarray(Xreal_ff).T[1],Ureal_ff]).T
np.savetxt("ff_measured_data.csv", measured_csv_data, delimiter=',', header="time,pos,vel,tau", comments="")

# Show video
Video(exp_video_path)

## 6c. PID+Gravity Compensation Experiment

**Question: Even though the pendulum hardware and simulation behavior is different, why does the gravity compensation controller still work?**

In [ ]:
x0 = 0.0
tf = 5 # Final time (s)
dt = 0.01 # Time step (s)
# runs the controller on hardware
Treal_grav_pid_con, Xreal_grav_pid_con, Ureal_grav_pid_con, Ureal_grav_pid_con_des, exp_video_url, exp_video_path = pendulum.run_on_hardware_cloud(user_token = user_token, 
                                                                                        tf = tf,
                                                                                        dt = dt,
                                                                                        x0 = x0,
                                                                                        controller = grav_pid_con
                                                                                                       )

Xdes_grav_pid_con = np.ones(int(tf/dt))*np.pi

# Measured Position
plt.figure
plt.plot(Treal_grav_pid_con, np.asarray(Xreal_grav_pid_con).T[0])
plt.plot(Treal_grav_pid_con, Xdes_grav_pid_con)
plt.xlabel("Time (s)")
plt.ylabel("Position (rad)")
plt.title("Position (rad) vs Time (s)")
plt.show()

# Measured Torque
plt.figure
plt.plot(Treal_grav_pid_con, Ureal_grav_pid_con)
plt.plot(Treal_grav_pid_con, Ureal_grav_pid_con_des)
plt.xlabel("Time (s)")
plt.ylabel("Torque (Nm)")
plt.legend(['Measured Torque', 'Desired Torque'])
plt.show()
# Show video
Video(exp_video_path)

## 6d. Energy Shaping Experiment

In [ ]:
x0 = 0.0
tf = 5 # Final time (s)
dt = 0.01 # Time step (s)

energy_shaping_controller.b = 0.0004

Treal_es, Xreal_es, Ureal_es, Ureal_es_des, exp_video_url, exp_video_path = pendulum.run_on_hardware_cloud(user_token = user_token, 
                                                                                        tf = tf,
                                                                                        dt = dt,
                                                                                        x0 = x0,
                                                                                        controller = energy_shaping_controller
                                                                                                       )
                                                                                                       

# Measured Position
plt.figure
plt.plot(Treal_es, np.asarray(Xreal_es).T[0])
plt.xlabel("Time (s)")
plt.ylabel("Position (rad)")
plt.title("Position (rad) vs Time (s)")
plt.show()

# Measured Velocity
plt.figure
plt.plot(Treal_es, np.asarray(Xreal_es).T[1])
plt.xlabel("Time (s)")
plt.ylabel("Velocity (rad/s)")
plt.title("Velocity (rad/s) vs Time (s)")
plt.show()

# Measured Torque
plt.figure
plt.plot(Treal_es, Ureal_es)
plt.plot(Treal_es, Ureal_es_des)
plt.xlabel("Time (s)")
plt.ylabel("Torque (Nm)")
plt.title("Torque (Nm) vs Time (s)")
plt.show()

# Export the data to csv file
measured_csv_data_es = np.array([Treal_es, np.asarray(Xreal_es).T[0], np.asarray(Xreal_es).T[1],Ureal_es]).T
np.savetxt("es_measured_data_es.csv", measured_csv_data_es, delimiter=',', header="time,pos,vel,tau", comments="")
# Show video
Video(exp_video_path)

In [ ]:
pendulum.plot_energy()